In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/4_linear_regression.ipynb)

# Rudiments of Machine Learning: Learning a Continuous Function

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 23.04.2026
+ **Authors:** [Dr. Benedikt Zönnchen](https://bzoennchen.github.io/Pages/)

When we talk about *artificial intelligence* we mostly refer to *machine learning*. All of the progress that you hear about in the mainstream media is duo to *machine learning* or, even more specifically, *deep learning*. In this notebook, we are going to strip away the hype and see ML-models as the **realization of certain mathematical operations**. Later, in another notebook, we will extend on this by introducing a probabilistic viewpoint. Here, we will take a journey through three different ways to solve the same problem, moving from 2,000-year-old math to modern-day *machine learning*.

⛳ **Our Learning Goals**

By the end of this notebook, you will understand the three pillars of machine learning:

1. Representation: How does a computer "see" a relationship? (linear vs. non-linear).
2. Evaluation: How does a computer "know" it is wrong? (the loss function).
3. Optimization: How does it get better? (The "slow walk" of *gradient descent*).

We will do everything via `Python` code, thereby letting the computer do the calculation.

😰 **Prerequisites: What do you need to know?**

You do not need to be a computer scientist or a mathematician to follow this exercise. 
However, the notebook will be most helpful if you are comfortable with: 

+ High school algebra: Specifically, the *concept of a line* ($y = mx + b$) and a *function*, e.g. $h(x) = x^2$, and its *derivative* $'h(x) = 2x$. You should understand that $x$ is our input and $y$ is the result / output.
+ Basics of linear algebra: It helps do know about *matrix-vecotor multiplication* and systems of *linear equations*.
+ Basic `Python` literacy: You don't need to write complex code from scratch, but you should be comfortable reading basic `Python` (like variables, loops, and lists) and clicking the `Run` button in a Jupyter cell.
+ Mental openness to "approximation": In school, we are taught that math is either right or wrong. In machine learning, math is often about being "mostly right" or "close enough."

**⚠️ Before you start — important notes for Google Colab:**
- **Save your work first:** Go to *File → Save a copy in Drive* to save this notebook to your Google Drive. After that, Colab will auto-save your progress. You can also download your work at any time via *File → Download → Download .ipynb*❗
- **Run cells in order, from top to bottom.** Each cell can only use variables defined in cells that have already been run❗ 
- **Got an unexpected error?** Go to *Runtime → Restart session and run all* to start fresh❗ 
- **Colab disconnects after ~30 minutes of inactivity**, which erases all your variables. If this happens, go to *Runtime → Restart session and run all*. Your saved notebook in Drive is not affected❗ 

In [ ]:
#@title install required packages, download test files, initialize Otter

# install packages
%pip install otter-grader==5.5.0
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install torch
%pip install torchinfo
%pip install torchviz
%pip install torchview
%pip install graphviz
%pip install matplotlib
%pip install seaborn

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os
    os.makedirs('tests', exist_ok=True)
    api = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A1_introduction/tests"
    for f in requests.get(api).json():
        open(f"tests/{f['name']}", 'w').write(requests.get(f['download_url']).text)

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('4_linear_regression.ipynb')

---

🗣 **Definitions** 

+ *Artificial Intelligence (AI)* refers to the overarching concept of creating machines or software capable of performing tasks that typically require human intelligence. This includes everything from basic logic and "if-then" rules to complex problem-solving. 
+ *Machine learning (ML)* is a subset of AI. The defining characteristic of ML is that the computer isn't explicitly programmed with rules for every scenario. Instead, it uses algorithms and statistical models to find patterns in data and make decisions based on those patterns. 
+ *Deep learning (DL)* is a subset of machine learning. It is inspired by the structure of the human brain, specifically artificial neural networks. The "deep" refers to the many layers of these networks that process data. It is exceptionally good at handling unstructured data like images, sound, and text.

---

## 12 (Linear) Regression

### 12.1 Preparation

First we import all `Python` packages we need. These imports basically copy the code that is behind each package into our working environment such that the Python interpreter finds the code of these packages.

+ ``torch`` is the PyTorch libary which contains all the code to construct artificial neural networks.
+ ``torch.nn as nn`` means that from now on we can write ``nn`` when we actually refer to ``torch.nn``. You can think of the dot ``.`` like moving into a subdirectory
+ ``numpy`` is a package to for numerical scientific computing (not specific to machine learning)
+ ``matplotlib`` is a package to plot
+ ``seaborn`` is a package to get even better plots, it is based on ``matplotlib``.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchview import draw_graph

import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Configure our plotting engine to get nice visualiziations
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

### 12.2 Problem Statement: Linear Regression

In this example, we are going to explore the core principles of *supervised* machine learning by learning a very simple function

$$h(x) = 2x + 5$$

given some data 

$$\{(x_1,y_1), (x_2, y_2), (x_3, y_3), \ldots \},$$

also sometimes called *observations*. In other words, given some observations, we want to find a function that

1. *fits* the data such that our observations can be reproduced, e.g. $h_\theta(x_1) = y_1$,
2. *has predictive power*, meaning that unseen data (a new $x$) lead "good" predictions ($h_\theta(x) = y$).

We call $x$ an *independent* variable and $y$ a *dependent* variable, because in our perspective, $y$ depends on $x$.

---

🗣 **Hint:** Any mainstream AI-system be it ChatGPT, StableDiffusion, Midjourney or any other tool is based on a trained model which is "just" the realization of a very complex mathematical function!

---

We keep the example so simple in the hope that you can focus on the methods and the new vocabulary instead of the example itself. The hope is that you can connect the new with the familiar.

The problem itself is a classical *regression task* i.e. where the goal is to predict a continuous, numerical value. In our special case we cheat a bit because we already know the solution $h$. What we are contructing is another function $h_\theta$ such that 

$$\forall x \in [a;b]: \Vert h(x) - h_\theta(x) \Vert < \epsilon.$$

In other words, we want to contruct a parameterized function $h_\theta$ with parameters $\theta$ such that it is the difference $\Vert h(x)-h_\theta(x)\Vert$ is very small. Note that $h$ and $h_\theta$ are two different mathematical objects. In the end $h_\theta(x)$ will be realized by an artificial neural network for which we optimize its parameters $\theta$. $h_\theta$ is often called *hypothesis* or (trained) *model*.

### 12.2 Solving the Problem

Let us define $h$ in Python. Note that we cheat here: The task is to find $h$ but we use $h$ to generate fake data / fake observations.

In [ ]:
# This defines our function h(x) = 2x + 5 in Python
# Using a lambda (one-liner shorthand) — equivalent to the def below:
# h = lambda x: 2*x + 5

def h(x):
    return 2*x + 5

# Quick check
print(f'h(1) = {h(1)},  expected 7')
print(f'h(3) = {h(3)},  expected 11')

#### 12.2.1 Solving a Linear Equation

---

🗣 **Remarks:** Sections *12.2.1* and *12.2.2* use matrices and linear algebra to solve the problem exactly.
If you are not comfortable with matrices, feel free to **skim or skip to Section 12.2.3** — gradient descent (Section 12.2.1) is the approach used in modern AI and does not require matrix math.

---

First we generate our data. Usually we would get such data from elsewhere. $x$ could be the size of a flat while $y$ its price. Thus, given our observation of the housing market, we want to construct a function that predicts the cost of a flat dependent on its size.

In [ ]:
# Our data points (x, y)
x = np.array([1, 3])    # np.array is a special data (a special list) ready for numerical computations
y = h(x)                # computes y = np.array([h(1), h(3)]), i.e. applys h to each component in x

In [ ]:
x

In [ ]:
y

So currently our observation is $\{ (1,7), (3,11) \}$.

Let's plot the data to see how the function might look:

In [ ]:
sns.scatterplot(x=x, y=y, label="Data Points")
sns.lineplot(x=x, y=y, color="red", label="Trend line")
plt.show()

Looking at the data, we see that this is probably a *linear function*. Thus, we assume

$$h_\theta(x) = w_0 + w_1 \cdot x$$

for some $w_0, w_1$ because any linear function with only one variable $x$ can be expressed in this manner!

---

🗣 **Hint:** At this point an important change in your thinking should happen: Usually $x$ is our variable. You know functions like 

$$g(x) = x^2 + 3x - 1$$

and you know how to compute, e.g. its derivative $\frac{dg}{dx}$ which, in this case, is 

$$\frac{dg}{dx} = \frac{d (x^2 + 3x - 1)}{dx} = g'(x) = 2x + 3.$$

$h_\theta(x)$ is just a function and $x$ is still its variable but we search $w_0$ and $w_1$. While we know $h_\theta(x_i) = y_i$ for some pairs of $(x_i, y_i)$, we do not know $\theta$, that is, $w_0, w_1$!

---

Given the data $\{ (1,7), (3,11) \}$, we know that $h_\theta(1) = 7$ and $h_\theta(3) = 11$ should hold. Therefore, we want to solve a very familiar problem. We have the following equations:

\begin{equation}
\begin{split}
w_0 + w_1 \cdot 1 &= 7 \\
w_0 + w_1 \cdot 3 &= 11
\end{split}
\end{equation}

In matrix form this can be written as 

$$
\underbrace{\begin{bmatrix}
1 & 1 \\
1 & 3
\end{bmatrix}}_{A} \cdot 

\underbrace{\begin{bmatrix}
w_0 \\
w_1
\end{bmatrix}}_{\theta} = 
\underbrace{\begin{bmatrix}
7 \\
11
\end{bmatrix}}_{y}
$$

Luckily we can solve this equation exactly by computing the inverse of $A$:

$$A \cdot \theta = y \iff A^{-1}A \theta = A^{-1}y \iff \theta = A^{-1}y$$

This gives us the **exact** answer instantly. There is no "learning," no "epochs," and no "error". It is a perfect mathematical calculation.

---

🖍 **Exercise 12.1:** Solve the linear equation (1) by hand meaning find $w_0$ and $w_1$ such that $w_0 + w_1 \cdot 1 = 7$ AND $w_0 + w_1 \cdot 3 = 11$.

---

**Solution:** Let's do the calculation by hand. Given

\begin{align*}
w_0 + w_1 \cdot 1 &= 7  \quad \ \, (I)\\
w_0 + w_1 \cdot 3 &= 11 \quad (II)
\end{align*}

we subtract $(II)$ by $(I)$ to get

\begin{align*}
w_0 + w_1 \cdot 1 &= 7  \quad (I)\\
w_1 \cdot 2 &= 4 \quad (II)
\end{align*}

From $(II)$ we get $w_1 = 2$ and if we put it into $(I)$ we get 

$$w_0 + 2 = 7$$

thus 

$$w_0 = 5.$$

Let's now use `Python` and our computer to do it:

In [ ]:
# Construct the matrix A
A = np.vstack([np.ones(len(x)), x]).T
A

In [ ]:
# Compute the inverse of A
A_inv = np.linalg.inv(A)
A_inv

In [ ]:
# Solve for theta:
theta = A_inv @ y # @ instead of * because we want a matrix vector multiplication!
theta

In [ ]:
print(f"Classical solution found: h(x) = {theta[1]:.2f}x + {theta[0]:.2f}")

We are done! We found our first parameters $w_0$ and $w_1$ of our very first model $h_\theta$!

---

🗣 **Hint:** As you might noticed, we had 2 variables $w_0, w_1$ and 2 equations such that everything worked out, i.e. $A$ was **invertable**! This was only the case because we constructed the data ourselves. Usually we have much more noisy observations than parameters. In fact, usually we need much more observations than parameters. In this case, $A$ is not invertable and we can not find an exact fit.

---

Let us create more faked data but this time the data is *noisy*!

#### 12.2.2 Least Squares Approximation

In [ ]:
np.random.seed(42) # For reproducibility: this makes sure we generate the same pseudorandom values each time we execude the code
N = 200     # We want N (fake) observations

# The "true" signal
x = np.linspace(0, 10, N)
y_obs = 2 * x + 5

# Adding Gaussian (normal) noise
noise = np.random.normal(0, 2.5, N) # mean=0, std_dev=2.5
y = y_obs + noise

sns.scatterplot(x=x, y=y, label="Data Points")
plt.show()

We still can write down a system of linear equations

$$A \cdot \theta = y$$

but this time $A$ and $y$ have 200 rows:

$$
\underbrace{\begin{bmatrix}
1 & x_{1} \\
1 & x_{2} \\
1 & x_{3} \\
\vdots & \vdots \\
1 & x_{200}
\end{bmatrix}}_{A} \cdot 

\underbrace{\begin{bmatrix}
w_0 \\
w_1
\end{bmatrix}}_{\theta} = 
\underbrace{\begin{bmatrix}
y_1 \\
y_2 \\
y_3 \\
\vdots \\
y_{200}
\end{bmatrix}}_{y}.
$$

Because we cannot solve this exactly, we should ask: What is a "good" approximation?

The answer is that we want to minimize the error we make in our (future) predictions. And without further considerations, we value each pair equally important. Thus, we look at all pairs of data points $(x_i, y_i)$ and look for $h_\theta$ such that the **mean squared error** is minimized. Therefore, we search a $\theta$ such that

\begin{align*}
L_{\text{MSE}} = \min_{\theta}\frac{1}{N} \sum_{i=1}^N (h_\theta(x_i) - y_i)^2 &= \min_{\theta}  \frac{1}{N} \left[ (h_\theta(x_1) - y_1)^2 + (h_\theta(x_2) - y_2)^2 + \ldots + (h_\theta(x_N) - y_N)^2 \right] \\
&= \min_{w_0, w_1} \frac{1}{N} \left[ (w_1 \cdot x_1 + w_0 - y_1)^2 + (w_1 \cdot x_2 + w_0 - y_2)^2 + \ldots + (w_1 \cdot x_N + w_0 - y_N)^2 \right] \\
&= \min_{\theta} J(\theta).
\end{align*}

In our case $N = 200$ holds. $h_\theta(x_i)$ is the prediction of our model $h_\theta$ and $y_i$ the "true" value of the observation. $J(\theta)$ is called *cost function*. For the cost function $x_i, y_i$ are fixed and $\theta$ are its variables. Using *gradient descent* we could try to solve this by a costly step by step optimization:

$$\theta_k = \theta_{k-1} - \eta \cdot \nabla J(\theta_{k-1}).$$

Luckily, even if there are many more observations than variables, we can solve the whole optimization problem in one calculation using standard matrix vector multiplication **IF** $h_\theta$ is linear (linear algabra is just beautiful!). To find the best possible $\theta$, we use the **normal equation**:

$$\theta = (A^\top A)^{-1} A^\top y$$

---

🗣 **Hint:** While $A$ is not invertable, $A^\top A$ is.

---

In [ ]:
# Construct the matrix A
A = np.vstack([np.ones(len(x)), x]).T
A.shape

In [ ]:
# Instead of the inverse for A we compute the inverse for (A^T A)
AA_inv = np.linalg.inv(A.T @ A)
AA_inv

In [ ]:
# Solve for theta:
theta = AA_inv @ A.T @ y # @ instead of * because we want a matrix vector multiplication!
theta

In [ ]:
print(f"Approximation found: h(x) = {theta[1]:.2f}x + {theta[0]:.2f}")

In [ ]:
# Define our model
h_theta = lambda x: theta[0] + x * theta[1]

In [ ]:
# Plot data and our model prediction (the fit)
sns.scatterplot(x=x, y=y, label="Data Points")
sns.lineplot(x=x, y=h_theta(x), color="red", label=r"$h_\theta(x)$")
plt.show()

Let us compute the error we make for our observations:

In [ ]:
error = h_theta(x) - y
mse = (error.T @ error) / N # alternative np.mean(error**2)
print(f"MSE: {mse}") 


Looks good! We have found a good model.

Let us now check if *gradient descent*, using the MSE (mean squared error), leads us to the same solution.

---

🖍 **Exercise 12.2:** What happens if there is a points that is very far away from all the other observations?
Define this point and see what happens with the error and the $h_\theta$ (the red line). Note that the following plot does not contain the outlier itself. 

---

In [ ]:
x_outlier = 10    # TODO: fill in some number
y_outlier = -500  # TODO: fill in some number

y_with_outlier = np.concatenate([y, np.array([y_outlier])])
x_with_outlier = np.concatenate([x, np.array([x_outlier])])
A_with_outlier = np.vstack([np.ones(len(x_with_outlier)), x_with_outlier]).T
AA_inv_outlier = np.linalg.inv(A_with_outlier.T @ A_with_outlier)
theta_outlier = AA_inv_outlier @ A_with_outlier.T @ y_with_outlier
h_theta_outlier = lambda x: theta_outlier[0] + x * theta_outlier[1]

# prints the error
error = h_theta_outlier(x_with_outlier) - y_with_outlier
mse = (error.T @ error) / N
print(f"MSE: {mse}") 

sns.scatterplot(x=x, y=y, label="Data Points with an Outlier")
sns.lineplot(x=x_with_outlier, y=h_theta_outlier(x_with_outlier), color="red", label=r"$h_\theta(x)$")
plt.show()

We call these points *outliers*. They might be caused by unwanted artefacts but they might also be caused by "being different from the norm".

#### 12.2.3 Gradient Descent

We reached an optimal solution using linear algebra. Now, let's see how an **artificial neural network (ANN)** finds it. Instead of solving a linear equation, an ANN is usually trained by *gradient descent*.

**The Logic**
1.  Start at a random solution for $\theta$: The model guesses random values for $\theta$, i.e. for $w_0$ and $w_1$.
2.  Calculate the error: It looks at the **MSE** (mean squared error):
    $$J(\theta) = \frac{1}{N} \sum_{i=1}^{N} ( (w_1 \cdot x_i + w_0) - y_i )^2$$
3.  Find the slope (gradient): It calculates the derivative of the error. This tells the model which direction is "downhill" toward the minimum error.
4.  The nudge: It updates the weights:
    $$\theta_{\text{new}} = \theta_{\text{old}} - \eta \cdot \frac{\partial J}{\partial \theta}$$

Don't be afraid of the Greek symbols! The *learning rate* is usually denoted as $\eta$ and
$\frac{\partial J}{\partial \theta}$ is the derivative of $J$ with respect to $\theta$. Since $\theta$ consists of 2 variables, we have to compute the derivative with respect to $w_0$ **and** $w_1$. Thus the result is a vector, i.e. the gradient.

---

🖍 **Exercise 12.3:** Compute the gradient $\nabla J(\theta_{k-1})$ assuming that we only have two observations $\{ (1,7), (3,11) \}$ (this is our example from above). In this case

$$J(\theta) = \frac{1}{2} \cdot \left[ (w_1 \cdot 1 + w_0 - 7)^2 + (w_1 \cdot 3 + w_0 - 11)^2 \right]$$

---

**Solution:** 

\begin{align*}
\frac{dJ}{dw_0} &= (w_1 \cdot 1 + w_0 - 7) + (w_1 \cdot 3 + w_0 - 11) = 4 w_1 + 2 w_0 - 18 \\
\frac{dJ}{dw_1} &= 1 \cdot (w_1 \cdot 1 + w_0 - 7) + 3 \cdot (w_1 \cdot 3 + w_0 - 11) = 10 w_1 + 4 w_0 - 40
\end{align*}

so the gradiant is 

$$\nabla J(\theta_{k-1}) = \begin{bmatrix} 4 w_1 + 2 w_0 - 18 \\ 10 w_1 + 4 w_0 - 40 \end{bmatrix}.$$

More generally, the gradient of 

$$J(\theta) = \frac{1}{N} \sum_{i=1}^{N} ( (w_1 \cdot x_i + w_0) - y_i )^2$$

is

\begin{align*}
\frac{dJ}{dw_0} &= \frac{2}{N} \sum_{i=1}^{N} (w_1 x_i + w_0 - y_i) \\
\frac{dJ}{dw_1} &= \frac{2}{N} \sum_{i=1}^{N} x_i (w_1 x_i + w_0 - y_i).
\end{align*}

---

🖍 **Exercise 12.4:** Assuming 

$$\theta_{k-1} = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix} = \begin{bmatrix} 6 \\ 3 \end{bmatrix}$$

and given the gradient you computed above: compute $\theta_{k}$ for $\eta = 0.1$. What do you observe? How should $w_0, w_1$ change in the next step?


---

**Solution:** We first compute the gradient for the given falues of $\theta_{k-1}$ by plugging the values in. This gives us

$$\nabla J(\theta_{k-1}) = \begin{bmatrix} 4 \cdot 3 + 2 \cdot 6 - 18 \\ 10 \cdot 3 + 4 \cdot 6 - 40 \end{bmatrix} = \begin{bmatrix} 6 \\ 14 \end{bmatrix}$$

Then we can compute

$$\theta_{k} = \theta_{k-1} - \eta \cdot \nabla J(\theta_{k-1}) = \begin{bmatrix} 6 \\ 3 \end{bmatrix} - 0.1 \cdot \begin{bmatrix} 6 \\ 14 \end{bmatrix} = \begin{bmatrix} 5.4 \\ 1.6 \end{bmatrix}$$

We are getting closer to the solution $w_0 = 5, w_1 = 2$. $w_0$ and $w_1$ decreased. In the next step $w_0$ should further decrease and $w_1$ should increase.


Solving the problem by *gradient descent* means to repeat the calculation hundreds of times. It is **computationally expensive** compared to the classical way, but it is the way we usually train an artificial neural network.

In [ ]:
def train(x, y, epochs=100):
    # 1. Initialize parameters randomly
    w_0 = np.random.randn()
    w_1 = np.random.randn()

    learning_rate = 0.01 #0.1 learnig rate is usually to big

    N = len(x)

    # Store history for visualization
    history = []

    # 2. The manual training loop
    for epoch in range(epochs):
        # a. Make a prediction
        y_pred = w_1 * x + w_0
        
        # b. Calculate the error (MSE)
        error = y_pred - y
        loss = np.mean(error**2)
        
        # c. Calculate gradients (the "slope" of the error)
        grad_w_0 = (2/N) * np.sum(error)
        grad_w_1 = (2/N) * np.sum(x * error)
        
        # d. Update Parameters (The "Nudge")
        w_0 = w_0 - (learning_rate * grad_w_0)
        w_1 = w_1 - (learning_rate * grad_w_1)
        
        history.append(loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss (MSE) {loss:.4f}, w_0: {w_0:.2f}, w_1: {w_1:.2f}")
    return w_0, w_1, history

w_0, w_1, history = train(x, y)
final_error = (w_1 * x + w_0) - y
final_loss = np.mean(final_error**2)
print(f"\nFinal loss (MSE) {final_loss}")
print(f"\nFinal result: y = {w_1:.2f}x + {w_0:.2f}")

In [ ]:
# Plot data and our model prediction (the fit)
sns.scatterplot(x=x, y=y, label="Data Points")
sns.lineplot(x=x, y=w_0 + x * w_1, color="red", label=r"$h_\theta(x)$")
plt.show()

---

🗣 **Definition:** In one (training) *epoch* the model has seen all the data once. This the number of *epochs* defines how often the model sees the data during training.

---

Let us also plot the development of the loss, that is, the loss with respect to the optimization step.

In [ ]:
# 1. Create a figure
plt.figure(figsize=(10, 5))

# We use range(len(history)) for the x-axis (Epochs)
sns.lineplot(x=range(len(history)), y=history, color="firebrick", linewidth=2.5)

# 3. Aesthetics
plt.title("The 'Learning Curve': Evolution of Error")
plt.xlabel("Iteration (Epochs)")
plt.ylabel("Mean Squared Error (Loss)")

# Adding a 'Starting' and 'Ending' point annotation for clarity
plt.annotate(f'Start Loss: {history[0]:.2f}', xy=(0, history[0]), xytext=(20, 5), textcoords='offset points', arrowprops=dict(arrowstyle='->'))
plt.annotate(f'End Loss: {history[-1]:.2f}', xy=(len(history)-1, history[-1]), xytext=(20, 20), textcoords='offset points', arrowprops=dict(arrowstyle='->'))

sns.despine()
plt.show()

---

🖍 **Exercise 12.5:** Let us plot everything after the 5th iteration such that we can better see if there is potential to train longer. Do you think we should train for longer, i.e. increase the number of epochs?

---

In [ ]:
# 1. Create a figure
plt.figure(figsize=(10, 5))

history_after_5 = history[5:]

# We use range(len(history)) for the x-axis (Epochs)
sns.lineplot(x=range(len(history_after_5)), y=history_after_5, color="firebrick", linewidth=2.5)

# 3. Aesthetics
plt.title("The 'Learning Curve': Evolution of Error")
plt.xlabel("Iteration (Epochs)")
plt.ylabel("Mean Squared Error (Loss)")

# Adding a 'Starting' and 'Ending' point annotation for clarity
plt.annotate(f'Start Loss: {history_after_5[0]:.2f}', xy=(0, history_after_5[0]), xytext=(20, 5), textcoords='offset points', arrowprops=dict(arrowstyle='->'))
plt.annotate(f'End Loss: {history_after_5[-1]:.2f}', xy=(len(history_after_5)-1, history_after_5[-1]), xytext=(20, 20), textcoords='offset points', arrowprops=dict(arrowstyle='->'))

sns.despine()
plt.show()

---

🖍 **Exercise 12.6:** The variable `history` stores the training loss at each step. Assign the **first** (initial) loss value to `initial_loss` and the **last** (final) loss value to `final_loss`.

🗣 **Hint:** In Python, `my_list[0]` returns the first element and `my_list[-1]` returns the last element.*

---

In [ ]:
initial_loss = ...
final_loss   = ...

print(f'Loss at start of training: {initial_loss:.4f}')
print(f'Loss at end of training:   {final_loss:.4f}')
print(f'Total improvement:         {initial_loss - final_loss:.4f}')

In [ ]:
grader.check("q126")

---

🖍 **Exercise 12.7:** Change the variable `epochs` such that you are satisfied with the result. Justify your choice.

---

In [ ]:
epochs = ...
w_0, w_1, histor_improved = train(x, y, epochs=epochs)

In [ ]:
grader.check("q127")

#### 12.2.4 Training an Artificial Neural Network

Using an artificial neural network (ANN) to solve the problem might seem like an overkill but ANNs can in fact be very simple.
For the input $x$ of layer $k$ the output of the layer is the component-wise application of a simple activation function $a$ of the result of a matrix vector multiplication:

\begin{equation*}
a(W \cdot x).
\end{equation*}

$W$ is a matrix. $W \cdot x$ is a vector and $a(W \cdot x)$ means to apply the function to each component of the vector. Thus the result is another vector. 

---

🗣 **Hint:** Sometimes you might see $x^\top \cdot W$ instead of $W \cdot x$. Which gives you the same result but instead of a column-vector you get a row-vector.

---

If we choose 

\begin{equation*}
W = \theta = \begin{bmatrix} w_0 \\ w_1 \end{bmatrix}
\end{equation*}

(a 2-times-1 matrix) and $a(x) = x$

\begin{equation*}
a(x) = x
\end{equation*}

(the identity function), then we can model $h_\theta$ via the most simple artificial neural network!

As soon as we enter the realm of training our own ANN, we rely on the ``torch`` package (also called PyTorch).
You can think of it as an extension of `numpy` but instead of matrices and vectors (both are `array`s) it works with multidimensional `tensor`s and offers all the basic components to build your own ANN.

We looked at the scalar vector multiplication in the last notebook.

In [ ]:
import numpy as np

a = np.array([[1,2,3,4]])
b = np.array([[-1, 0, 0, 1]])

dot_product = a @ b.T
dot_product

We can do the same with `torch`

In [ ]:
import torch

a = torch.tensor([[1,2,3,4]])
b = torch.tensor([[-1, 0, 0, 1]])

dot_product = a @ b.T
dot_product

It is easy to convert a `numpy` array into a `torch` tensor and vice versa:

In [ ]:
numpy_a = np.array([[1,2,3,4]])
torch_a = torch.tensor(numpy_a)

print(type(numpy_a))
print(type(torch_a))

numpy_b = torch_a.numpy()

print(type(numpy_b))

But we can do more. We can work with 

$$n \times m \times k \times \ldots$$

tensors!

In [ ]:
# Here we define a 6 x 3 x 4 tensor which we can multiply with any 4 x a x b x ... tensor from the right or with any ... a x b x 6 tensor from the left.
my_tensor = torch.tensor(
  [
    [[1,2,3,4], [1,2,3,4], [1,2,3,4]], 
    [[0,0,1,1], [0,0,1,1], [0,0,1,1]],
    [[1,2,3,4], [1,2,3,4], [1,2,3,4]], 
    [[0,0,1,1], [0,0,1,1], [0,0,1,1]],
    [[1,2,3,4], [1,2,3,4], [1,2,3,4]], 
    [[0,0,1,1], [0,0,1,1], [0,0,1,1]],
  ])

print(my_tensor.shape)

my_other_tensor = torch.tensor(
  [
    [1,1,1],
    [0,0,0],
    [0,1,1],
    [1,0,0]
  ])

print(my_other_tensor.shape)

my_new_tensor = my_tensor @ my_other_tensor

print(my_new_tensor.shape)

Ok, what is going on here?

`my_tensor` has shape `(6, 3, 4)` -- think of it as 6 matrices, each of size 3x4.
`my_other_tensor` has shape `(4, 3)` -- a single 2D matrix. PyTorch's `@` operator performs batched matrix multiplication. It treats the last two dimensions as the matrix dimensions and broadcasts over any leading dimensions.
So effectively, each of the 6 slices of shape `(3, 4)` is multiplied by the `(4, 3)` matrix independently.
The result my_new_tensor has shape `(6, 3, 3)`.

These massive amount of matrix multiplication is what is required in machine learning and GPUs are optimized for this type of operations, they are currently the kings of machine learning hardware. Most of the frustration of training your own model in `torch` comes from defining the correct shape of your tensors.

In [ ]:
# 1. Prepare data for PyTorch (converting from NumPy)
# x and noisy_y were created in the previous step
# PyTorch expects the input format: (number of observations, number of features) => here: (200, 1)
# but x has shape (200,) meaning it is a 1D vector not a 2D 200x1 vector (this is always confusing!)
# .view(-1, 1) solves this problem and reshapes the data.
# .view(200, 1) also works
X_tensor = torch.tensor(x, dtype=torch.float32).view(-1, 1)     # float32 means use 32 bits for each number
Y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)     # "tensor" is just a fancy mathematical name for a multidimensional array

print(f"shape of x: {x.shape}")
print(f"shape of x: {X_tensor.shape}")

# 2. Define a simple linear model (y = w_1 * x + w_0)
# This is a neural network with 1 input, 1 output, and NO activation
model = nn.Linear(1, 1) # model paramized hypothesis h_theta

# 3. Setup the optimizer and Loss
criterion = nn.MSELoss() # we use the mean squared error as be fore
optimizer = optim.SGD(model.parameters(), lr=0.01) # stochastic gradient descent instead of gradient descent

# 4. Training loop
epochs = 500
for epoch in range(epochs):
    # Forward pass: Compute predicted y by passing x to the model
    prediction = model(X_tensor)
    loss = criterion(prediction, Y_tensor)
    
    # Backward pass: Find the gradients
    optimizer.zero_grad()   # clear the gradient(s) computed by the last iteration
    loss.backward()         # compute the new gradient(s)
    
    # Update: Nudge the weights downhill
    optimizer.step()

# 5. Extract results
w_1_learned = model.weight.item()
w_0_learned = model.bias.item()

In [ ]:
print(f"--- Comparison ---")
print(f"Target:    y = 2.00x + 5.00")
print(f"ANN found:  y = {w_1_learned:.2f}x + {w_0_learned:.2f}")

final_error = (w_1_learned * x + w_0_learned) - y
final_loss = np.mean(final_error**2)
print(f"\nFinal loss (MSE) {final_loss}")

We can get a summary of our ANN:

In [ ]:
from torchinfo import summary
summary(model, input_size=(10, 1))

We can also visualize the network:

In [ ]:
from torchview import draw_graph

with torch.no_grad():
  model_graph = draw_graph(model, input_data=X_tensor)
model_graph.visual_graph

---

🗣 **Hint** Our ANN has 2 parameters that got optimized. A large language model like `Llama 4` has **400 billion** parameters. Training such models a lot of money / energy!

---

## 13 A Real World Problem (California Housing)

Now that we understand how a single neuron learns a line, let's see what happens when we have **8 inputs** and **thousands of rows** of data. 

For this we use the **California Housing dataset**. It consists of 20640 data points with 8 numeric, predictive attributes and the target - thus overall 9 features:

- ``MedInc``:        median income in block group
- ``HouseAge``:      median house age in block group
- ``AveRooms``:      average number of rooms per household
- ``AveBedrms``:     average number of bedrooms per household
- ``Population``:    block group population
- ``AveOccup``:      average number of household members
- ``Latitude``:     block group latitude
- ``Longitude``:     block group longitude

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block group is the smallest geographical unit for which the U.S.
Census Bureau publishes sample data (a block group typically has a population
of 600 to 3,000 people).

We want to predict house prices. In this dataset, the relationship between "income" and "house price" isn't a perfect straight line - it's a complex, noisy cloud. We will use a multi-layer perceptron (MLP) to find the patterns that a simple ruler cannot.

In the following code we load the data from the ``sklearn`` Python package which offers some sample datasets.

In [ ]:
# Load the dataset
data = fetch_california_housing()

Then we convert the data into a so called Pandas ``dataframe`` which is like an Excel sheet / a table of data that we can easily view and manipulate. It helps us to get an overview over the data.

In [ ]:
# Convert data to a dataframe
df = pd.DataFrame(data.data, columns=data.feature_names)
# target is what we wanna predict i.e. the y's everything else belongs to the x's
df['MedHouseVal'] = data.target 

After executing this code ``df`` represents the dataframe. It is the object that currently contains all the data. We can display parts of the dataframe.

In [ ]:
# Show the first 5 rows
print("The First 5 Rows of California Housing Data:")
df.head()

We can also get a specific column (called feature) by addressing it by its name:

In [ ]:
df['HouseAge']

We can also get a multiple columns by addressing them by name. We need to use a list ``[]`` of column names to do so. In this case, the result is also a dataframe:

In [ ]:
df[['HouseAge', 'MedInc']]

There are many useful possibilities to work dataframes. For example, we can compute the average (*mean*) of ``HouseAge``:

In [ ]:
df['HouseAge'].mean()

In [ ]:
df.columns

We can get a feeling for the data by plotting a histogram for each feature:

In [ ]:
df.hist(figsize=(12, 10), bins=30, edgecolor="black")
plt.subplots_adjust(hspace=0.7, wspace=0.4)

The distribution shows that house counts generally decrease as the median value increases, with the notable exception of a spike at the highest valuation. This pattern appears anomalous and may suggest a ceiling effect in the data collection process - specifically, that all values exceeding 5 were capped at that limit. Similarily the income also peaks at the highest value.

Visualizing the data is no longer that easy because we have overall 9 features and we can not draw a point cloud in a 9-dimensional space. But we can plot pairs of features. For example, one can expect that ``MedInc`` and ``MedHouseVal`` are *correlated*. Let's investigate:

In [ ]:
# Visualize the non-linearity (Income vs Price)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(1000), x='MedInc', y='MedHouseVal', alpha=0.5)
plt.title("Is this a straight line? (Median Income vs. House Value)")
plt.show()

---

🖍 **Exercise 13.1:** Now that you have explored the California Housing dataset, answer the following two questions by filling in the code below:

1. What is the **average median income** (`MedInc`) across all districts? Assign it to `avg_income`.
2. How many **data points** (districts) are in the dataset? Assign the count to `n_samples`.

🗣 **Hint:** Use `.mean()` on the column (as shown above) and `len(df)` for the row count.*

---

In [ ]:
avg_income = ...
n_samples  = ...

print(f'Average median income: {avg_income:.4f}')
print(f'Number of data points: {n_samples}')

In [ ]:
grader.check("q131")

---

🖍 **Exercise 13.2:** The `MedHouseVal` column holds the median house value for each district (in units of $100,000). Assign the **minimum** value to `min_price` and the **maximum** value to `max_price`.

🗣 **Hint:** Use `.min()` and `.max()` — these work on any pandas column.

---

In [ ]:
min_price = ...
max_price = ...

print(f'Minimum house value: {min_price:.2f}  (= ${min_price * 100_000:,.0f})')
print(f'Maximum house value: {max_price:.2f}  (= ${max_price * 100_000:,.0f})')

In [ ]:
grader.check("q132")

---

🖍 **Exercise 13.3:** How many districts have an **average house age greater than 30 years**? Assign the count to `n_old_districts`.

🗣 **Hint:** `df[df['HouseAge'] > 30]` selects all rows where the condition is True. Then use `len(...)` to count them.*

---

In [ ]:
n_old_districts = ...
print(f'Districts with average house age > 30 years: {n_old_districts}')

In [ ]:
grader.check("q133")

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 13.4 (Reflection):** We predicted **house prices** using features like location, income, and house age. Now imagine you want to build a model to predict the **auction price of an artwork**. Which **three features** (inputs) would you include in your model? Briefly explain your choices in the cell below.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

Let us now just use all features to predict the ``MedHouseVal`` and let us use an artificial neural network to do so.

This time we split our dataset into a *training set* and a *test set*.
We only train by using the training set meaning that our model will only see the training set and never the test set. The reason to do so is to test our model for data it never has seen before thus to prevent it from just *memorizing* the data. We also say that we want to prevent **overfitting**.

In [ ]:
# 1. Data Preparation
X = data.data
y = data.target.reshape(-1, 1)

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Since the features exist on vastly different scales - for instance, ``MedInc`` ranges from 0 to 15 while ``Population`` reaches into the thousands - we *standardize* the data. Without this, features with larger magnitudes would disproportionately dominate the learning process; their larger values would produce larger gradients, potentially causing the optimization algorithm to ignore all other features.

The standardization realizes the following formula:

$$z = \frac{x - \mu}{\sigma}$$

where $z$ is the standardized value, $x$ the non-standardized, $\mu$ the mean and $\sigma$ the standard deviation of the respective feature.

After this computation all features are centered around $0$ and have standard deviation equal to $1$.

In [ ]:
print(f"Means per feature: {X_train.mean(axis=0)}")
print(f"Standard deviation per feature: {X_train.std(axis=0)}")

In [ ]:
# Scale the data (Crucial for Neural Networks!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
print(f"Means per feature: {X_train.mean(axis=0)}")
print(f"Standard deviation per feature: {X_train.std(axis=0)}")

To use ``torch`` we have to convert the ``numpy`` array into a ``torch`` array.

In [ ]:
# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

With the data prepared, we must now define our model architecture and determine the appropriate number of parameters. A more complex model is not only more computationally expensive to train, but it also implies a more intricate underlying structure or set of causal relationships within our observations.

**PyTorch building blocks — a quick reference**

| Term | What it means |
|------|---------------|
| `nn.Module` | Base class for all neural network models in PyTorch |
| `nn.Linear(in, out)` | A fully-connected layer: learns a weight matrix + bias |
| `F.relu(x)` | Activation function — sets negative values to 0 (the "hinge") |
| `forward(x)` | Defines how data flows through the model |
| `optim.Adam(...)` | An optimizer — smarter gradient descent |
| `loss.backward()` | Computes gradients automatically (backpropagation) |
| `optimizer.step()` | Updates the weights using the gradients |

In [ ]:
# 2. Define the model architecture
class HousingModel(nn.Module):
    def __init__(self):
        super(HousingModel, self).__init__()
        # We start with 8 inputs and expand to 32 "clues"
        self.fc1 = nn.Linear(8, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1) # Final price prediction

    def forward(self, x):
        x = F.relu(self.fc1(x)) # "Hinge" 1
        x = F.relu(self.fc2(x)) # "Hinge" 2
        return self.fc3(x)      # Final output

# 3. Create the model
model = HousingModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# 4. Train the model via the training loop
losses = []
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# 5. Evaluation the model by testing it for unseen data
model.eval()
with torch.no_grad():
    predictions = model(X_test_t)
    test_error = criterion(predictions, y_test_t)

print(f"Final Test Error (MSE): {test_error.item():.4f}")

At this stage, we should proceed more critically. It is possible that ``MedHouseVal`` has little to no relationship with certain features, or that some provided variables are impractical to collect. For instance, requiring ``MedInc`` (median income) may not always be feasible, even though it likely correlates strongly with property values.

Let us now look more closly at the data by plotting all pairs. We only use the first ``n`` data points because plotting them all is computational expensive.

In [ ]:
n = 1000
features_to_plot = ['MedInc', 'HouseAge', 'AveRooms', 'AveOccup']
sns.pairplot(df[:n],  
            x_vars=df.columns, 
            y_vars=['MedHouseVal'], 
            kind='reg', 
            aspect=0.8,
            plot_kws={'scatter_kws': {'alpha': 0.1, 's': 10}})

``MedHouseVal`` is correlated with ``MedInc`` as well as ``AveRooms`` (number of rooms). However, correlation is a linear relationship.

---

🖍 **Exercise 13.5:** If you think about the geography of California, what might be a good indicator for the value of a house?

---

Let us a heat map of the house value based on the geographical values!

In [ ]:
ax = sns.scatterplot(
    data=df,
    x="Longitude",
    y="Latitude",
    size="MedHouseVal",
    hue="MedHouseVal",
    palette="viridis",
    alpha=0.5,
)

ax.set_aspect('equal', adjustable='box')

The geographic distribution clearly mirrors the coastline of California, where property values are notably higher than in inland areas.

Let us try to learn this heatmap!
We only use 2 imputs i.e. ``Latitude`` and ``Longitude`` to predict ``MedHouseVal``.

In [ ]:
# 1. Data Preparation
X = df[['Latitude', 'Longitude']].to_numpy()
y = data.target.reshape(-1, 1)

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data (Crucial for Neural Networks!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# 2. Define the model architecture
class HousingModelGeo(nn.Module):
    def __init__(self):
        super(HousingModelGeo, self).__init__()
        # We start with 8 inputs and expand to 32 "clues"
        self.fc1 = nn.Linear(2, 512)
        self.fc2 = nn.Linear(512, 32)
        self.fc3 = nn.Linear(32, 1) # Final price prediction

    def forward(self, x):
        x = F.relu(self.fc1(x)) # "Hinge" 1
        x = F.relu(self.fc2(x)) # "Hinge" 2
        return self.fc3(x)      # Final output

# 3. Create the model
model = HousingModelGeo()
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# 4. Train the model via the training loop
losses = []
for epoch in range(500):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# 5. Evaluation the model by testing it for unseen data
model.eval()
with torch.no_grad():
    predictions = model(X_test_t)
    test_error = criterion(predictions, y_test_t)

print(f"Final Test Error (MSE): {test_error.item():.4f}")

Let us visualize how we can think of our artificial neural networks as a sequence of layers:

In [ ]:
with torch.no_grad():
  model_graph = draw_graph(model, input_data=X_train_t)
model_graph.visual_graph

In [ ]:
with torch.no_grad():
    # Get predictions and reshape them back to the 2D grid shape
    preds_test = model(X_test_t)
    preds_train = model(X_train_t)

In [ ]:
df_test = pd.DataFrame({"Longitude": X_test_t[:,1], "Latitude": X_test_t[:,0], 'MedHouseVal': preds_test[:,0]})
df_train = pd.DataFrame({"Longitude": X_train_t[:,1], "Latitude": X_train_t[:,0], 'MedHouseVal': preds_train[:,0]})

# Create a figure with 1 row and 3 columns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

titles = ["Ground Truth (Original)", "Training Predictions", "Test Predictions"]
datasets = [df, df_train, df_test]

for ax, df_i, title in zip(axes, datasets, titles):
    sns.scatterplot(
        data=df_i,
        x="Longitude",
        y="Latitude",
        size="MedHouseVal",
        hue="MedHouseVal",
        palette="viridis",
        alpha=0.5,
        ax=ax,
        legend=False # Cleaner without 3 legends
    )
    ax.set_title(title)
    #ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

## Well done!

You've learned the basics of *supervised machine learning* in `Python`.

In particular, you should now be able to:

 - have an intuitive understanding of the machine learning pipeline (starting from gathering observation to predicting unseen data)
 - have an intuitive understanding of role of *observations*, *gradients*, *learning rate*, an *epoch*, *gradient descent*, *parameters of an ANN* within the machine learning pipeline;
 - understand that it important to think about which input features are usful to predict the output;
 - know about the concept of a *tensor*
 - know what a *regression task* is.

In the next notebook we learn about another supervised machine learning task, that is, *classification*. Go to the next notebook:

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/5_classification.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>